In [ ]:
# 后面清洗数据时用到 shutil.move，这里先导入
# （原来这里错误地 import 了 transformers 的 ImageClassifierOutput 并对未定义的 learn 调用，已删除）
import shutil


In [ ]:
# 初次运行 fastbook 的环境配置（下载依赖 / 做设置）
import fastbook
fastbook.setup_book()


In [ ]:
# 导入 fastbook 与 fastai 视觉小部件（图形化清洗器等 GUI 组件）
# （原来这里有一行 [[chapter_production]]，那是书里的 markdown 标记，放在代码里会报 NameError，已删除）
from fastbook import *
from fastai.vision.widgets import *


In [ ]:
# 测试搜索：抓取 "grizzly bear" 的图片链接，看能返回多少张
ims = search_images_ddg('grizzly bear')   # 修正：原来是 'grizzly.bear'
len(ims)


In [ ]:
# 三类熊：灰熊 / 黑熊 / 玩具熊
bear_types = 'grizzly', 'black', 'teddy'
path = Path('bears')


In [ ]:
# 为每一类各建一个子文件夹并下载对应图片
if not path.exists():
    path.mkdir()
    for o in bear_types:
        dest = (path/o)
        dest.mkdir(exist_ok=True)
        results = search_images_ddg(f'{o} bear')
        download_images(dest, urls=results)


In [ ]:
# 收集所有下载好的图片路径
fns = get_image_files(path)
fns


In [ ]:
# 找出损坏 / 无法打开的图片
failed = verify_images(fns)
failed


In [ ]:
# 删除损坏图片；注意传的是 Path.unlink 这个函数本身
failed.map(Path.unlink)   # 修正：原来写成 Path.unlink()，加了括号会先调用、报错


In [ ]:
# 定义数据管道：图片 -> 类别，8:2 划分，父文件夹名做标签，统一 Resize 到 128
bears = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=Resize(128)
)


In [ ]:
# 用 bears 这个蓝图构建 DataLoaders
dls = bears.dataloaders(path)


In [ ]:
# 看一批验证集图片
dls.valid.show_batch(max_n=4, nrows=1)


In [ ]:
# 换成 Squish（压扁）缩放方式再看效果
bears = bears.new(item_tfms=Resize(128, ResizeMethod.Squish))
dls = bears.dataloaders(path)
dls.valid.show_batch(max_n=4, nrows=1)


In [ ]:
# 换成随机裁剪（数据增强），每次取图片不同区域
bears = bears.new(item_tfms=RandomResizedCrop(128, min_scale=0.3))
dls = bears.dataloaders(path)
dls.train.show_batch(max_n=4, nrows=1, unique=True)


In [ ]:
# 用 batch 级数据增强（旋转 / 亮度等），同一张图看多种增强
bears = bears.new(item_tfms=Resize(128), batch_tfms=aug_transforms(mult=2))
dls = bears.dataloaders(path)
dls.train.show_batch(max_n=4, nrows=2, unique=True)


In [ ]:
# 同上，展示更多张
bears = bears.new(item_tfms=Resize(128), batch_tfms=aug_transforms(mult=2))
dls = bears.dataloaders(path)
dls.train.show_batch(max_n=8, nrows=2, unique=True)


In [ ]:
# 最终训练用配置：随机裁剪到 224 + 标准增强
bears = bears.new(item_tfms=RandomResizedCrop(224, min_scale=0.5), batch_tfms=aug_transforms())
dls = bears.dataloaders(path)


In [ ]:
# resnet18 迁移学习，微调 4 轮
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(4)


In [ ]:
# 混淆矩阵：看每一类被分对 / 分错的情况
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix()


In [ ]:
# 展示损失最大的 5 张（模型最"困惑"的样本）
interp.plot_top_losses(5, nrows=1, figsize=(17, 4))


In [ ]:
# 图形化清洗器：手动标记要删除或改类别的图片
cleaner = ImageClassifierCleaner(learn)
cleaner


In [ ]:
# 执行清洗器里的标记：删除选中的、把改类别的移动到目标文件夹
for idx in cleaner.delete(): cleaner.fns[idx].unlink()
for idx, cat in cleaner.change(): shutil.move(str(cleaner.fns[idx]), path/cat)


In [ ]:
# —— 猫狗分类（用文件名首字母大小写判断是不是猫）——
from fastai.vision.all import *
def is_cat(x): return x[0].isupper()


In [ ]:
# 用文件名规则加载数据
path = untar_data(URLs.PETS)/'images'
dls = ImageDataLoaders.from_name_func(
    '.',
    get_image_files(path), valid_pct=0.2, seed=42, label_func=is_cat,
    item_tfms=Resize(192)
)


In [ ]:
# 看一批猫狗图片
dls.show_batch()


In [ ]:
# resnet18 微调 3 轮
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(3)


In [ ]:
# 导出训练好的模型到 model.pkl，供推理 / 部署使用
learn.export('model.pkl')   # 修正：原来拼成 exprot


In [ ]:
# —— 用 Gradio 做一个推理小页面 ——
from fastai.vision.all import *
import gradio as gr
def is_cat(x): return x[0].isupper()


In [ ]:
# 载入一张测试图并生成缩略图
im = PILImage.create('dog.jpg')
im.thumbnail((192, 192))
im


In [ ]:
# 载入导出的模型并预测
learn = load_learner('model.pkl')
learn.predict(im)


In [ ]:
# 把预测概率整理成 {类别: 概率} 字典，供 Gradio 展示
categories = ('Dog', 'Cat')
def classify_image(img):
    pred, idx, probs = learn.predict(img)
    return dict(zip(categories, map(float, probs)))


In [ ]:
# 测试分类函数
classify_image(im)


In [ ]:
# 搭建 Gradio 界面（注意：gr.inputs / gr.outputs 是 Gradio 3.x 的旧写法，4.x 已移除）
image = gr.inputs.Image(shape=(192, 192))
label = gr.outputs.Label()   # 修正：原来是小写 label()
examples = ['dog.jpg', 'cat.jpg', 'dunno.jpg']
intf = gr.Interface(fn=classify_image, inputs=image, outputs=label, examples=examples)
intf.launch(inline=False)


In [ ]:
# 取出底层 PyTorch 模型
m = learn.model


In [ ]:
# 模型的全部参数（权重张量列表）
ps = list(m.parameters())


In [ ]:
# 第一层参数的形状
ps[0].shape


In [ ]:
# 第一层参数的具体数值
ps[0]


In [ ]:
# 把 notebook 转成 python 脚本（旧版 nbdev API）
from nbdev.export import notebook2script


In [ ]:
# 将 app.ipynb 导出为脚本
notebook2script('app.ipynb')
